In [ ]:
params <- list(
    ps_raw = "data/temp/ps_raw.rds"
)
set.seed(8092003)

In [ ]:
library(phyloseq)
library(dplyr)
library(microbiome)
library(DESeq2)
library(ggplot2)
library(ggpubr)
library(ggrepel)
# library(pheatmap)
library(tibble)
library(tidyr)
library(RColorBrewer)
library(Maaslin2)

In [ ]:
source("workflow/scripts/plots/plots_pallet.R")

In [ ]:
ps_raw <- readRDS(params$ps_raw)
ps_core <- microbiome::core(
    ps_raw,
    detection = 1,
    prevalence = 0.05
)
meta <- data.frame(sample_data(ps_raw))

# DESeq2

In [ ]:
deseq_counts <- phyloseq_to_deseq2(ps_raw, ~ lpb_tertile + batch + d_sex)
deseq_counts <- estimateSizeFactors(deseq_counts, type = "poscounts")

In [ ]:
dds <- DESeq(deseq_counts, test="Wald", fitType="parametric")

In [ ]:
resultsNames(dds)

In [ ]:
res <- results(dds, name="lpb_tertile_Низкий.ЛПБ_vs_Высокий.ЛПБ")
summary(res)

In [ ]:
res <- results(dds, cooksCutoff = FALSE)
alpha <- 0.01
sigtab <- res[which(res$padj < alpha), ]
sigtab <- cbind(as(sigtab, "data.frame"), as(tax_table(ps_raw)[rownames(sigtab), ], "matrix"))
head(sigtab)

In [ ]:
scale_fill_discrete <- function(palname = "Set1", ...) {
    scale_fill_brewer(palette = palname, ...)
}
color_cond <- sigtab$phylum
y_cond <- sigtab$genus

x <- tapply(sigtab$log2FoldChange, color_cond, function(x) max(x))
x <- sort(x, TRUE)
sigtab$Phylum <- factor(as.character(color_cond), levels=names(x))
# Genus order
x <- tapply(sigtab$log2FoldChange, y_cond, function(x) max(x))
x <- sort(x, TRUE)
sigtab$genus <- factor(as.character(y_cond), levels=names(x))

options(repr.plot.width = 8, repr.plot.height = 12)
ggdotchart(
    sigtab,
    y = "log2FoldChange",
    x = "genus",
    group = "phylum",
    color ="phylum",
    sorting = "descending",
    palette = my_palette(length(unique(sigtab$phylum))),
    add = "segments",
    rotate = T,
    dot.size = 6,
) + 
    geom_hline(yintercept = 0, color = "lightgray")


In [ ]:
res_df <- as.data.frame(res) %>%
    rownames_to_column("ASV") %>%
    arrange(padj)

# Удаление NA
res_df <- res_df %>%
    filter(!is.na(padj))

# Категории значимости
res_df <- res_df %>%
    mutate(
        significance = case_when(
            padj < alpha & log2FoldChange > 1 ~ "Up",
            padj < alpha & log2FoldChange < -1 ~ "Down",
            TRUE ~ "NS"
        )
    )

p_volcano <- ggplot(
    res_df,
    aes(
        x = log2FoldChange,
        y = -log10(padj),
        color = significance
    )
) +
    geom_point(alpha = 0.7, size = 2) +

    scale_color_manual(values = c(
        "Up" = "firebrick",
        "Down" = "steelblue",
        "NS" = "grey70"
    )) +

    geom_vline(xintercept = c(-1, 1),
               linetype = "dashed") +

    geom_hline(yintercept = -log10(alpha),
               linetype = "dashed") +

    geom_text_repel(
        data = subset(res_df, padj < alpha),
        aes(label = ASV),
        size = 3,
        max.overlaps = 20
    ) +

    theme_bw() +

    labs(
        title = "Differential abundance volcano plot",
        x = "log2 Fold Change",
        y = "-log10 adjusted p-value"
    )
options(repr.plot.width = 8, repr.plot.height = 8)
p_volcano

# Maaslin2

In [ ]:
counts_tab <- data.frame(otu_table(ps_core))
if (taxa_are_rows(ps_core)) {
    counts_tab <- t(counts_tab)
}


In [ ]:
ps_core

In [ ]:
set.seed(8092003)
mslr <- Maaslin2(
    counts_tab,
    meta,
    "data/temp/maaslin",
    fixed_effects = c("lpb_tertile", "d_sex"),
    random_effects = c("batch"),
    reference = c('lpb_tertile,Средний ЛПБ'),
    min_prevalence = 0,
    normalization = 'TMM',
    transform = 'LOG',
    standardize = F,
    cores = 5)

In [ ]:
mslr.sign.lpb <- mslr$results[mslr$results$qval <= alpha]
mslr.sign.lpb <- cbind(as(mslr.sign.lpb, "data.frame"), as(tax_table(ps_core)[mslr.sign.lpb$feature, ], "matrix"))
head(mslr.sign.lpb)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 8)
ggdotchart(
    mslr.sign.lpb,
    y = "coef",
    x = "genus",
    group = "phylum",
    color ="phylum",
    sorting = "descending",
    palette = my_palette(length(unique(mslr.sign.lpb$phylum))),
    add = "segments",
    rotate = T,
    dot.size = 6,
) + 
    geom_hline(yintercept = 0, color = "lightgray")